# GraphiSense: Single-Image Inference Demonstration

This notebook demonstrates the end-to-end GraphiSense pipeline:
1. Loading a handwriting sample image.
2. Polarity-aware Otsu binarization.
3. Extracting the 9 reduced static features.
4. Running inference with the pre-trained Random Forest model artifact.
5. Displaying model probabilities and transparency measurements.

> **Disclaimer**: GraphiSense is an academic research and screening-support prototype. It does not provide medical or clinical diagnoses.

In [ ]:
import sys
from pathlib import Path

# Add repository root to sys.path
repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import cv2
from src.preprocessing.image_loader import load_image_with_grayscale
from src.preprocessing.binarization import binarize_handwriting
from src.models.inference import predict_handwriting

dataset_root = repo_root / 'data' / 'raw'
print('GraphiSense inference module loaded successfully.')


## 1. Select Input Handwriting Image

In [ ]:
# Choose a test sample (can be modified to any image path)
sample_path = dataset_root / 'Potential Dysgraphia' / 'PD (1).jpg'
print('Evaluating sample:', sample_path.name)


## 2. Execute Prediction & Feature Extraction

In [ ]:
result = predict_handwriting(sample_path)

print('Predicted Class Label:', result['predicted_label'])
print('Model Probabilities:', result['class_probabilities'])
print('\nExtracted Features Used (Reduced Static Features):')
for feat_name, feat_val in result['features_used'].items():
    print(f'  {feat_name:<30} = {feat_val}')


## 3. Visualize Original, Binary Mask, and Distance Transform

In [ ]:
orig_bgr, gray = load_image_with_grayscale(sample_path)
orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
mask = binarize_handwriting(gray, method='otsu', polarity='dark_background')
dist_map = cv2.distanceTransform(mask, cv2.DIST_L2, 5)

fig, axes = plt.subplots(3, 1, figsize=(12, 7))
axes[0].imshow(orig_rgb)
axes[0].set_title(f'Original Image: {sample_path.name}')
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title('Binarized Foreground Mask (Handwriting = 255, Background = 0)')
axes[1].axis('off')

im_dist = axes[2].imshow(dist_map, cmap='inferno')
fig.colorbar(im_dist, ax=axes[2], fraction=0.03, pad=0.02, label='Inscribed Radius (px)')
mean_sw = result['features_used']['stroke_width_mean']
axes[2].set_title(f'Medial Axis Distance Transform (Mean Stroke Width: {mean_sw:.2f} px)')
axes[2].axis('off')

plt.suptitle(f"Predicted: {result['predicted_label']} | P(PD) = {result['class_probabilities']['PD']:.1%}", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Clinical Disclaimer

> **Important**: The predicted class is an experimental statistical baseline reflecting patterns in the Mendeley Dysgraphia Handwriting Dataset. It is not an approved medical diagnostic test and does not substitute for clinical neuropsychological assessment.